# AI Food Audit Assistant - Colab


In [ ]:
# Setup Colab environment
import sys
import os
from pathlib import Path

# Detect if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Install required packages for Colab
    !pip install -q pandas matplotlib seaborn requests huggingface_hub
    !pip install -q llama-cpp-python --no-cache-dir 2>/dev/null || pip install -q llama-cpp-python-cuda
else:
    !pip install -q -r requirements.txt

print("✓ Dependências instaladas com sucesso")

In [ ]:
# Create project structure and upload files
from pathlib import Path

# Define working directory
if IN_COLAB:
    work_dir = Path('/content')
else:
    work_dir = Path.cwd()

# Create necessary directories
for directory in ['data', 'outputs', 'outputs/graficos', 'outputs/relatorios', 'models', 'logs']:
    (work_dir / directory).mkdir(parents=True, exist_ok=True)

os.chdir(work_dir)
print(f"✓ Diretório de trabalho: {os.getcwd()}")

# Create Python modules
config_code = '''from pathlib import Path

ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data"
OUTPUTS_DIR = ROOT_DIR / "outputs"
GRAPHICS_DIR = OUTPUTS_DIR / "graficos"
REPORTS_DIR = OUTPUTS_DIR / "relatorios"
MODELS_DIR = ROOT_DIR / "models"
LOGS_DIR = ROOT_DIR / "logs"

for d in [DATA_DIR, OUTPUTS_DIR, GRAPHICS_DIR, REPORTS_DIR, MODELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
MODEL_FILE = "qwen2.5-0.5b-instruct-q5_k_m.gguf"

CONFORME_VALUES = {"sim", "s", "true", "1", "ok", "yes", "conforme"}
REQUIRED_COLUMNS = [
    "Data", "Setor", "Auditor", "Item auditado", "Categoria",
    "Conforme", "Observação", "Ação corretiva", "Responsável", "Prazo"
]
'''

utils_code = '''import logging
from typing import Iterable
import pandas as pd
from pathlib import Path

LOGS_DIR = Path.cwd() / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

def setup_logger(name: str = "food_audit") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
        sh = logging.StreamHandler()
        sh.setFormatter(fmt)
        logger.addHandler(sh)
        fh = logging.FileHandler(LOGS_DIR / "app.log", encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    return logger

logger = setup_logger()

def ensure_columns(df: pd.DataFrame, cols: Iterable[str]) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes: {missing}")

def normalize_text(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.replace(r"\\s+", " ", regex=True).str.title()

def safe_dt(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce", dayfirst=True)

def topn(series: pd.Series, n: int = 5) -> dict:
    if series.empty:
        return {}
    return series.value_counts().head(n).to_dict()
'''

prompts_code = '''SYSTEM_PROMPT = """Você é um auditor especialista em qualidade e segurança de alimentos.
Analise os dados com objetividade, indique riscos, priorize ações e responda em português."""

EXEC_SUMMARY_PROMPT = """Gere um resumo executivo em até 120 palavras com base nos indicadores abaixo.

Conformidade geral: {conformidade}%
Total de não conformidades: {nc_count}
Setor com mais NC: {top_sector}
Categoria com mais falhas: {top_category}
Itens recorrentes: {recorrentes}

Inclua riscos e uma leitura executiva do cenário.
"""

TECH_PROMPT = """Gere uma análise técnica em até 180 palavras com foco em:
- riscos críticos
- provável causa raiz
- ações imediatas
- prevenção

Pontos críticos:
{critical_issues}
"""

ACTION_PROMPT = """Crie um plano de ação prático com prioridade 1, 2 e 3, em até 120 palavras.

Use como base:
- Conformidade: {conformidade}%
- Setor crítico: {top_sector}
- Categoria crítica: {top_category}
- Itens recorrentes: {recorrentes}
"""

QA_CHAT_PROMPT = """Responda à pergunta usando somente o contexto da auditoria.

Pergunta: {pergunta}

Contexto:
{contexto}

Responda em português, de forma objetiva e útil, até 90 palavras.
"""
'''

food_audit_code = '''from dataclasses import dataclass
from pathlib import Path
from typing import Dict
import pandas as pd
from config import CONFORME_VALUES, REQUIRED_COLUMNS, GRAPHICS_DIR, REPORTS_DIR, OUTPUTS_DIR
from utils import logger, ensure_columns, normalize_text, safe_dt, topn

@dataclass
class AuditResult:
    indicadores: Dict
    df: pd.DataFrame

class FoodAuditAnalyzer:
    def __init__(self, csv_path: str | Path):
        self.csv_path = Path(csv_path)
        self.df = pd.DataFrame()
        self.indicadores: Dict = {}

    def load(self) -> pd.DataFrame:
        self.df = pd.read_csv(self.csv_path, encoding="utf-8")
        ensure_columns(self.df, REQUIRED_COLUMNS)
        logger.info(f"CSV carregado: {len(self.df)} linhas")
        return self.df

    def clean(self) -> pd.DataFrame:
        df = self.df.copy()
        text_cols = ["Setor", "Auditor", "Item auditado", "Categoria", "Observação", "Ação corretiva", "Responsável"]
        for c in text_cols:
            df[c] = normalize_text(df[c])

        df["Conforme"] = df["Conforme"].astype(str).str.strip().str.lower()
        df["Conforme_Numerico"] = df["Conforme"].apply(lambda x: 1 if x in CONFORME_VALUES else 0)

        df["Data"] = safe_dt(df["Data"])
        df["Prazo"] = safe_dt(df["Prazo"])
        self.df = df
        logger.info("Dados limpos e normalizados")
        return df

    def indicators(self) -> Dict:
        df = self.df.copy()
        total = len(df) or 1
        nc_count = int((df["Conforme_Numerico"] == 0).sum())
        conformidade = round(((total - nc_count) / total) * 100, 2)

        nc = df[df["Conforme_Numerico"] == 0]
        setor_nc = topn(nc["Setor"])
        categoria_nc = topn(nc["Categoria"])
        itens = topn(nc["Item auditado"])
        recorrentes = {k: v for k, v in itens.items() if v > 1}

        temp = df.dropna(subset=["Data"]).copy()
        if not temp.empty:
            temp["Mes"] = temp["Data"].dt.to_period("M")
            evolucao = temp.groupby("Mes")["Conforme_Numerico"].agg(["count", "sum"])
            evolucao["nc_mensal"] = evolucao["count"] - evolucao["sum"]
        else:
            evolucao = pd.DataFrame(columns=["count", "sum", "nc_mensal"])

        self.indicadores = {
            "conformidade": conformidade,
            "nc_count": nc_count,
            "total": total,
            "top_sector": next(iter(setor_nc), "N/A"),
            "top_category": next(iter(categoria_nc), "N/A"),
            "recorrentes": recorrentes,
            "setor_nc": setor_nc,
            "categoria_nc": categoria_nc,
            "evolucao": evolucao,
        }
        logger.info("Indicadores calculados")
        return self.indicadores

    def save_cleaned_csv(self) -> Path:
        out = OUTPUTS_DIR / "auditoria_tratada.csv"
        self.df.to_csv(out, index=False, encoding="utf-8")
        return out

    def chart_dashboard(self):
        import matplotlib.pyplot as plt
        from matplotlib.gridspec import GridSpec
        from matplotlib.patches import FancyBboxPatch
        import pandas as pd

        plt.style.use("seaborn-v0_8-whitegrid")

        fig = plt.figure(figsize=(13, 7), dpi=200)
        gs = GridSpec(
            3, 4,
            figure=fig,
            height_ratios=[1.0, 1.25, 1.35],
            width_ratios=[1.15, 1.15, 1.15, 1.15],
            hspace=0.55,
            wspace=0.35
        )

        def kpi_card(ax, x, y, w, h, title, value, color="#1f77b4"):
            ax.add_patch(
                FancyBboxPatch(
                    (x, y), w, h,
                    boxstyle="round,pad=0.02,rounding_size=0.03",
                    linewidth=1.0,
                    edgecolor="#d0d7de",
                    facecolor="white",
                    transform=ax.transAxes
                )
            )
            ax.text(x + 0.06, y + h - 0.20, title, transform=ax.transAxes,
                    fontsize=9, fontweight="bold", va="top", color="#444")
            ax.text(x + 0.06, y + 0.12, value, transform=ax.transAxes,
                    fontsize=16, fontweight="bold", va="bottom", color=color)

        # ----------------------------------------------------------
        # DONUT
        # ----------------------------------------------------------
        ax_donut = fig.add_subplot(gs[0, 0])
        conformidade = float(self.indicadores["conformidade"])
        nao = max(0.0, 100.0 - conformidade)

        ax_donut.pie(
            [conformidade, nao],
            startangle=90,
            counterclock=False,
            wedgeprops=dict(width=0.38, edgecolor="white"),
            autopct=lambda p: f"{p:.0f}%" if p >= 5 else ""
        )
        ax_donut.set_title("Conformidade", fontsize=11, pad=10)
        ax_donut.set_aspect("equal")

        # ----------------------------------------------------------
        # KPI CARDS
        # ----------------------------------------------------------
        ax_kpi = fig.add_subplot(gs[0, 1:4])
        ax_kpi.axis("off")

        card_w = 0.23
        gap = 0.02
        y = 0.12

        kpi_card(ax_kpi, 0.00, y, card_w, 0.68, "Conformidade", f"{conformidade:.1f}%")
        kpi_card(ax_kpi, card_w + gap, y, card_w, 0.68, "NC", str(self.indicadores["nc_count"]), "#d62728")
        kpi_card(ax_kpi, 2 * (card_w + gap), y, card_w, 0.68, "Setor Crítico", str(self.indicadores["top_sector"]), "#ff7f0e")
        kpi_card(ax_kpi, 3 * (card_w + gap), y, card_w, 0.68, "Categoria Crítica", str(self.indicadores["top_category"]), "#9467bd")

        # ----------------------------------------------------------
        # SETORES
        # ----------------------------------------------------------
        ax_setor = fig.add_subplot(gs[1, :2])
        setores = pd.Series(self.indicadores["setor_nc"]).head(5)

        if not setores.empty:
            ax_setor.barh(setores.index.astype(str), setores.values, height=0.55)
            ax_setor.invert_yaxis()
            ax_setor.tick_params(axis="y", labelsize=9)
            ax_setor.tick_params(axis="x", labelsize=8)
            ax_setor.grid(axis="x", linestyle="--", alpha=0.25)
        else:
            ax_setor.text(0.5, 0.5, "Sem dados suficientes", ha="center", va="center")

        ax_setor.set_title("Top Setores Críticos", fontsize=11, pad=8)
        ax_setor.set_xlabel("Quantidade")
        ax_setor.spines["top"].set_visible(False)
        ax_setor.spines["right"].set_visible(False)

        # ----------------------------------------------------------
        # CATEGORIAS
        # ----------------------------------------------------------
        ax_cat = fig.add_subplot(gs[1, 2:4])
        categorias = pd.Series(self.indicadores["categoria_nc"]).head(5)

        if not categorias.empty:
            ax_cat.barh(categorias.index.astype(str), categorias.values, height=0.55)
            ax_cat.invert_yaxis()
            ax_cat.tick_params(axis="y", labelsize=9)
            ax_cat.tick_params(axis="x", labelsize=8)
            ax_cat.grid(axis="x", linestyle="--", alpha=0.25)
        else:
            ax_cat.text(0.5, 0.5, "Sem dados suficientes", ha="center", va="center")

        ax_cat.set_title("Categorias Críticas", fontsize=11, pad=8)
        ax_cat.set_xlabel("Quantidade")
        ax_cat.spines["top"].set_visible(False)
        ax_cat.spines["right"].set_visible(False)

        # ----------------------------------------------------------
        # EVOLUÇÃO
        # ----------------------------------------------------------
        ax_line = fig.add_subplot(gs[2, :])
        evolucao = self.indicadores["evolucao"]

        if isinstance(evolucao, pd.DataFrame) and not evolucao.empty:
            ax_line.plot(
                evolucao.index.astype(str),
                evolucao["nc_mensal"],
                linewidth=2,
                marker="o",
                markersize=5
            )
            ax_line.set_ylabel("NC")
            ax_line.tick_params(axis="x", labelsize=8)
            ax_line.tick_params(axis="y", labelsize=8)
            ax_line.grid(True, axis="y", linestyle="--", alpha=0.25)
        else:
            ax_line.text(0.5, 0.5, "Sem dados suficientes", ha="center", va="center", fontsize=11)
            ax_line.set_axis_off()

        ax_line.set_title("Evolução das Não Conformidades", fontsize=11, pad=8)
        ax_line.spines["top"].set_visible(False)
        ax_line.spines["right"].set_visible(False)

        fig.suptitle("AI Food Audit Assistant", fontsize=15, fontweight="bold", y=0.98)
        plt.subplots_adjust(left=0.05, right=0.98, top=0.92, bottom=0.07)

        out = GRAPHICS_DIR / "dashboard_conformidade.png"
        plt.savefig(out, dpi=250, bbox_inches="tight")
        plt.close(fig)
        return out
    def executive_report(self) -> Path:
        txt = f"""RELATÓRIO EXECUTIVO

Conformidade: {self.indicadores['conformidade']}%
Total de NC: {self.indicadores['nc_count']}
Setor crítico: {self.indicadores['top_sector']}
Categoria crítica: {self.indicadores['top_category']}
Itens recorrentes: {list(self.indicadores['recorrentes'].keys())[:5]}
"""
        out = REPORTS_DIR / "relatorio_executivo.txt"
        out.write_text(txt, encoding="utf-8")
        return out

    def run(self) -> AuditResult:
        self.load()
        self.clean()
        self.indicators()
        self.save_cleaned_csv()
        self.chart_dashboard()
        self.executive_report()
        return AuditResult(self.indicadores, self.df)
'''

# Write module files
Path('config.py').write_text(config_code, encoding='utf-8')
Path('utils.py').write_text(utils_code, encoding='utf-8')
Path('prompts.py').write_text(prompts_code, encoding='utf-8')
Path('food_audit.py').write_text(food_audit_code, encoding='utf-8')

print("✓ Módulos Python criados com sucesso")

# Upload data file if in Colab
if IN_COLAB:
    from google.colab import files
    print("\n📤 Faça upload do arquivo 'exemplo_auditoria.csv' para a pasta 'data'")
    uploaded = files.upload()
    for filename in uploaded.keys():
        Path('data') / filename
        !mv {filename} data/
        print(f"✓ {filename} movido para data/")
else:
    print("✓ Usando arquivo de dados local: data/exemplo_auditoria.csv")

In [ ]:
# Download model from Hugging Face
from pathlib import Path
from huggingface_hub import hf_hub_download
from config import MODEL_REPO, MODEL_FILE, MODELS_DIR

try:
    print(f"📥 Baixando modelo: {MODEL_REPO}")
    model_path = Path(hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_FILE,
        local_dir=str(MODELS_DIR),
        local_dir_use_symlinks=False
    ))
    print(f"✓ Modelo baixado: {model_path}")
except Exception as e:
    print(f"⚠️ Erro ao baixar modelo: {e}")
    model_path = None

In [ ]:
# Load and analyze audit data
from food_audit import FoodAuditAnalyzer
from pathlib import Path

# Find CSV file
csv_files = list(Path('data').glob('*.csv'))
if not csv_files:
    raise FileNotFoundError("Nenhum arquivo CSV encontrado em data/")

csv_path = csv_files[0]
print(f"📊 Carregando: {csv_path.name}")

try:
    analyzer = FoodAuditAnalyzer(csv_path)
    result = analyzer.run()

    print("✓ Análise concluída com sucesso")
    print(f"\nIndicadores Principais:")
    print(f"  • Conformidade: {result.indicadores['conformidade']}%")
    print(f"  • Total de NC: {result.indicadores['nc_count']}")
    print(f"  • Setor crítico: {result.indicadores['top_sector']}")
    print(f"  • Categoria crítica: {result.indicadores['top_category']}")
except Exception as e:
    print(f"❌ Erro na análise: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Initialize LLM
from llama_cpp import Llama
from prompts import SYSTEM_PROMPT
from config import MODELS_DIR
from pathlib import Path

if model_path and model_path.exists():
    try:
        print("🤖 Inicializando modelo LLM...")
        llm = Llama(
            model_path=str(model_path),
            n_ctx=2048,
            n_threads=4,
            n_gpu_layers=0,
            verbose=False,
        )
        print("✓ Modelo LLM inicializado com sucesso")
    except Exception as e:
        print(f"⚠️ Erro ao inicializar LLM: {e}")
        llm = None
else:
    print("⚠️ Modelo não disponível. Pulando inicialização do LLM.")
    llm = None

In [ ]:
# Generate AI-powered insights
if llm is not None:
    from prompts import EXEC_SUMMARY_PROMPT, TECH_PROMPT, ACTION_PROMPT

    ind = result.indicadores
    context = analyzer.df.head(12).to_csv(index=False)

    # Executive Summary
    try:
        print("📝 Gerando Resumo Executivo...")
        summary_prompt = EXEC_SUMMARY_PROMPT.format(
            conformidade=ind['conformidade'],
            nc_count=ind['nc_count'],
            top_sector=ind['top_sector'],
            top_category=ind['top_category'],
            recorrentes=list(ind['recorrentes'].keys())[:5],
        )
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': summary_prompt}
        ]
        summary = llm.create_chat_completion(messages=messages, max_tokens=180, temperature=0.2)['choices'][0]['message']['content']
        print("✓ Resumo Executivo:\n", summary)
    except Exception as e:
        print(f"⚠️ Erro ao gerar resumo: {e}")
else:
    print("⚠️ LLM não disponível. Pulando geração de insights.")

In [ ]:
# Technical Analysis
if llm is not None:
    try:
        print("\n🔧 Gerando Análise Técnica...")
        tech_prompt = TECH_PROMPT.format(
            critical_issues='; '.join([f'{k} ({v})' for k, v in ind['recorrentes'].items()]) or 'Sem recorrências fortes'
        )
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': tech_prompt}
        ]
        tech = llm.create_chat_completion(messages=messages, max_tokens=220, temperature=0.2)['choices'][0]['message']['content']
        print("✓ Análise Técnica:\n", tech)
    except Exception as e:
        print(f"⚠️ Erro na análise técnica: {e}")
else:
    print("⚠️ LLM não disponível.")

In [ ]:
# Action Plan
if llm is not None:
    try:
        print("\n📋 Gerando Plano de Ação...")
        action_prompt = ACTION_PROMPT.format(
            conformidade=ind['conformidade'],
            top_sector=ind['top_sector'],
            top_category=ind['top_category'],
            recorrentes=list(ind['recorrentes'].keys())[:5],
        )
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': action_prompt}
        ]
        action = llm.create_chat_completion(messages=messages, max_tokens=160, temperature=0.2)['choices'][0]['message']['content']
        print("✓ Plano de Ação:\n", action)
    except Exception as e:
        print(f"⚠️ Erro ao gerar plano: {e}")
else:
    print("⚠️ LLM não disponível.")

In [ ]:
# Display Dashboard
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Image, display

dashboard_path = Path('outputs/graficos/dashboard_conformidade.png')
if dashboard_path.exists():
    print("📊 Dashboard de Conformidade:")
    display(Image(str(dashboard_path)))
else:
    print("⚠️ Dashboard não foi gerado.")

In [ ]:
# Q&A Chat with context
if llm is not None:
    from prompts import QA_CHAT_PROMPT

    try:
        print("\n❓ Respondendo pergunta...")
        question = 'Qual o principal risco e o que devo priorizar primeiro?'
        qa_prompt = QA_CHAT_PROMPT.format(pergunta=question, contexto=context)
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': qa_prompt}
        ]
        answer = llm.create_chat_completion(messages=messages, max_tokens=120, temperature=0.2)['choices'][0]['message']['content']
        print(f"Pergunta: {question}")
        print(f"Resposta:\n{answer}")
    except Exception as e:
        print(f"⚠️ Erro na resposta: {e}")
else:
    print("⚠️ LLM não disponível.")

In [ ]:
# Download results
from pathlib import Path

print("\n📥 Arquivos Gerados:")
report_path = Path('outputs/relatorios/relatorio_executivo.txt')
if report_path.exists():
    print(f"✓ {report_path.name}")
    with open(report_path) as f:
        print(f.read())

csv_path = Path('outputs/auditoria_tratada.csv')
if csv_path.exists():
    print(f"✓ {csv_path.name}")

if IN_COLAB:
    from google.colab import files

    print("\n📤 Iniciando download dos resultados...")
    if report_path.exists():
        files.download(str(report_path))
    if csv_path.exists():
        files.download(str(csv_path))
    dashboard_path = Path('outputs/graficos/dashboard_conformidade.png')
    if dashboard_path.exists():
        files.download(str(dashboard_path))
else:
    print("✓ Todos os arquivos estão em outputs/")